# 🎯 [DAT06_M10] 군수 월간 결산 및 보급 현황 보고서 원클릭 자동 생성 도구 최종 완성
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT06/dat06_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT06_M10 |
| **미션 주제** | 군수 월간 결산 및 보급 현황 보고서 원클릭 자동 생성 도구 최종 완성 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
미국 국방재무회계국(DFAS)은 반복적인 재무·정산 업무에 업무 자동화(RPA)를 도입해 사람이 하던 취합과 점검을 줄이는 사업을 추진해 왔다. 이 공개 사례의 학습 초점은 반복 업무 분해, 입력과 트리거, 자동 처리 순서, 오류 복구, 실행 로그, 시간 절감이다. 본 프로젝트는 이를 바탕으로 교육용 가상 부대 24개의 월간 결산 자료(물자 불출, 수리부속 소모, 탄약 정산, 장비 가동률)를 코드로 만들고, 파일 대신 메모리 안의 가상 문서와 표로 모의한다. 학습자는 업무 분해, 다형식 파싱, 검증 룰 엔진, 이상 원인 추천, 일관성 검증, 승인·반려 대시보드, 3단계 사람 결재선, 대량 배치 시험, AAR 보정, 최종 통합을 차례로 수행한다. 최종 목표는 월간 정산·보고·취합을 한 번에 실행하되 발간은 반드시 사람이 결재하도록 하는 원클릭 자동화 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> 💡 **[산출물(Deliverables) 디렉토리 경로 및 실습 안내]**
> - **아키텍처 관점의 표준 산출물:** 과제 명세에 표기된 src/connectors/..., docs/..., dist/... 등의 경로는 본 솔루션이 실무 배포 환경에서 갖추게 되는 **소프트웨어 표준 아키텍처 구조**를 의미합니다.
> - **노트북 중심 실습:** 별도의 폴더나 외부 파이썬 파일(.py)을 수작업으로 생성하실 필요가 없습니다. 본 노트북의 각 미션별 실습 셀을 AI 코파일럿과 함께 완성하고 검증하면 해당 산출물 요건이 자동으로 충족됩니다.
> - **미션 10 최종 통합 패키징:** 각 미션 셀에서 구현된 단위 모듈과 명세는 **[미션 10]** 실행 시 단일 소프트웨어 패키지(dist/dat06_package/)와 운영 매뉴얼(docs/dat06_final_manual.md)로 일괄 결합되어 자동 추출됩니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 군수 월간 결산 및 보급 현황 보고서 원클릭 자동 생성 도구 최종 완성
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

군수 월간 결산 및 보급 현황 보고서 원클릭 자동 생성 도구를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **수집부터 검증, 집계, 이상 점검, 보고서 생성, 결재 대기 분리까지 한 번에 실행되는 월간 결산 원클릭 도구를 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **환경 점검 후 CSV·JSON 혼합 자료를 수집하고 검증 룰과 집계, 이상 점검, 신뢰도 점수를 거쳐 24개 부대 보고서 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `dist/dat06_package/, README.md, docs/dat06_final_manual.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 상용 소프트웨어 수준의 완성도 및 조직 확산 가치

#### 미션 PASS 조건

- 정량 기준: 설치부터 실행까지 원클릭 설치 스크립트 완비, 매뉴얼 작성률 100%


- 진행 기준: 최종 배포본 생성 및 검수 완료


- 검증 방법: Clean 환경에서의 설치 및 초기화 테스트


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""환경 점검부터 4형식 자료 수집, 검증 룰 적용, 월간 결산 집계, 이상치 점검, 신뢰도 점수, 24개 부대 보고서 생성, 3단계 결재 대기 분리까지 한 번에 실행하는 원클릭 결산 도구를 완성하고 설치 스크립트 개요·매뉴얼 목차·자체 점검·브리핑 요약을 출력한다."""
import csv
import importlib.util
import io
import json
import sys
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 월간 결산 원장 생성기 (실제 부대명·품목·수량·재무 정보 없음)
# 가상 대대 24개 x (물자 불출 / 수리부속 소모 / 탄약 정산) 품목 원장과 장비 가동 현황을 만든다.
# 재고 산식: 기말 = 기초 + 수령 + 반납 - 불출,  다음 달 기초 = 이번 달 기말.
# ---------------------------------------------------------------------------
UNITS = [f"가상{i:02d}대대" for i in range(1, 25)]
ITEMS = {"물자불출": ["급식류", "피복류", "유류", "정비소모품"],
         "수리부속": ["필터류", "베어링류", "전장품", "타이어류"],
         "탄약정산": ["훈련탄A", "훈련탄B", "모의탄"]}
UNIT_CODE = {"급식류": "BX", "피복류": "EA", "유류": "L", "정비소모품": "EA", "필터류": "EA", "베어링류": "EA",
             "전장품": "EA", "타이어류": "EA", "훈련탄A": "RD", "훈련탄B": "RD", "모의탄": "RD"}
LEDGER_COLS = ["month", "unit", "cat", "item", "open", "recv", "issue", "ret", "close"]


def make_ledger(n_months=2, seed=60):
    """n_months개월 원장(list of dict). 둘째 달 기초는 첫째 달 기말과 같다."""
    rng = np.random.default_rng(seed)
    rows, prev = [], {}
    for m in range(n_months):
        for u in UNITS:
            for cat, items in ITEMS.items():
                for it in items:
                    base = int(rng.integers(80, 400))
                    op = prev.get((u, it), base * 3)
                    recv = int(rng.integers(0, base))
                    issue = int(rng.integers(base // 4, base // 2 + 1))
                    ret = int(issue * rng.uniform(0, 0.15))
                    cl = op + recv + ret - issue
                    rows.append({"month": f"2026-{m + 7:02d}", "unit": u, "cat": cat, "item": it,
                                 "open": op, "recv": recv, "issue": issue, "ret": ret, "close": cl})
                    prev[(u, it)] = cl
    return rows


def make_equip(n_months=2, seed=61):
    rng = np.random.default_rng(seed)
    out = []
    for m in range(n_months):
        for u in UNITS:
            hold = int(rng.integers(20, 60))
            out.append({"month": f"2026-{m + 7:02d}", "unit": u, "hold": hold, "operable": int(hold * rng.uniform(0.78, 0.97))})
    return out

LOG = []
def stage(name):
    def deco(fn):
        def run(*a):
            t0 = time.perf_counter()
            try:
                r = fn(*a); LOG.append((name, "OK", time.perf_counter() - t0)); return r
            except Exception as ex:                 # 한 단계가 실패해도 도구 전체가 죽지 않고 원인을 기록
                LOG.append((name, f"FAIL {repr(ex)[:40]}", time.perf_counter() - t0)); return None
        return run
    return deco

# ---------------- 0) 설치·환경 점검(원클릭 설치 스크립트 개요) ----------------
INSTALL = ["1. 가상 환경 생성 및 필수 모듈 설치(numpy, pandas, matplotlib)", "2. 환경 점검: 모듈·버전 확인", "3. 샘플 자료로 시험 실행", "4. 실행: run_monthly_close()", "5. 결과 확인: 보고서 24건, 결재 대기 목록"]
mods = ["numpy", "pandas", "matplotlib", "csv", "json"]
env_ok = sum(importlib.util.find_spec(m) is not None for m in mods)
print(f"[0] 환경 점검: Python {sys.version_info.major}.{sys.version_info.minor}, 필수 모듈 {env_ok}/{len(mods)} 확인 [교육용가상자료]")

# ---------------- 1) 수집(CSV + JSON 두 형식 혼합) ----------------
@stage("수집·파싱")
def collect():
    rows = [r for r in make_ledger(2) if r["month"] == "2026-08"]
    half = len(rows) // 2
    head = ["unit", "cat", "item", "open", "recv", "issue", "ret", "close"]
    csv_text = "\n".join([",".join(head)] + [",".join(str(r[h]) for h in head) for r in rows[:half]])
    json_text = json.dumps([{h: r[h] for h in head} for r in rows[half:]], ensure_ascii=False)
    a = list(csv.DictReader(io.StringIO(csv_text))) + json.loads(json_text)
    df = pd.DataFrame(a)
    for c in ("open", "recv", "issue", "ret", "close"):
        df[c] = pd.to_numeric(df[c])
    return df

# ---------------- 2) 검증 룰 ----------------
RULES = [("V1 산식", lambda d: d["open"] + d["recv"] + d["ret"] - d["issue"] == d["close"]), ("V2 음수", lambda d: (d[["open", "recv", "issue", "ret", "close"]] >= 0).all(axis=1)),
         ("V3 가용량", lambda d: d["issue"] <= d["open"] + d["recv"]), ("V4 반납", lambda d: d["ret"] <= d["issue"]), ("V5 부대", lambda d: d["unit"].isin(UNITS)),
         ("V6 품목", lambda d: d["item"].isin(UNIT_CODE))]
@stage("검증")
def check(df):
    bad = pd.Series(False, index=df.index)
    for _, f in RULES:
        bad |= ~f(df)
    return int(bad.sum())

@stage("집계")
def aggregate(df):
    return df.groupby(["unit", "cat"])[["issue", "ret", "close"]].sum().unstack("cat")

@stage("이상 점검")
def anomalies(df):
    prev = pd.DataFrame([r for r in make_ledger(2) if r["month"] == "2026-07"]).set_index(["unit", "item"])["issue"]
    ratio = df.set_index(["unit", "item"])["issue"] / prev.clip(lower=1)
    return ratio[ratio >= 4.0]

@stage("신뢰도")
def reliability(df, nbad):
    return round(100 - 100 * nbad / len(df), 1)

@stage("보고서 생성")
def build(df, agg):
    out = {}
    for u in UNITS:
        g = df[df["unit"] == u]
        out[u] = {"unit": u, "issue": int(g["issue"].sum()), "close": int(g["close"].sum()), "status": "결재 대기(기안)", "published": False}
    return out

df = collect()
nbad = check(df)
agg = aggregate(df)
anom = anomalies(df)
rel = reliability(df, nbad)
reports = build(df, agg)

# ---------------- 3) 결과 ----------------
print(f"[1] 수집 {len(df)}행(CSV+JSON 혼합), 검증 위반 {nbad}건, 이상 급증 후보 {len(anom)}건, 자료 신뢰도 {rel}점")
tbl = pd.DataFrame(reports).T[["issue", "close", "status"]].head(5)
print("[2] 부대별 결산 보고서(상위 5개 부대) - 모든 보고서는 결재 대기 상태")
print(tbl.to_string())
tot = agg["issue"].sum()
print("[3] 분류별 불출·소모 합계:", tot.astype(int).to_dict())
print("[4] 단계별 실행 로그:")
for n, st, t in LOG:
    print(f"   - {n:<8} {st:<5} {t * 1000:.1f}ms")
run_ms = sum(t for _, _, t in LOG) * 1000

# ---------------- 4) 자체 점검 ----------------
MANUAL = ["1. 개요·설치", "2. 입력 자료 양식", "3. 실행 방법(원클릭)", "4. 검증 룰 목록", "5. 이상 항목 처리", "6. 결재선·권한", "7. 오류 메시지 조치", "8. 배치 운영", "9. 개선 이력·문의"]
done_sec = len(MANUAL)
checks = [("환경 점검 5/5", env_ok == len(mods)), ("모든 단계 OK", all(s == "OK" for _, s, _ in LOG)), ("24개 부대 보고서 생성", len(reports) == 24),
          ("자동 발간 0건", not any(r["published"] for r in reports.values())), ("원장 산식 위반 0건", nbad == 0),
          ("실행 시간 1분 이내", run_ms < 60000), ("매뉴얼 9장 작성(작성률 100%)", done_sec == len(MANUAL)),
          ("설치 절차 5단계 완비", len(INSTALL) == 5)]
print("\n[5] 자체 점검")
for n, ok in checks:
    print(f"   {'PASS' if ok else 'FAIL'}  {n}")
npass = sum(ok for _, ok in checks)
print("설치 스크립트 개요:", " / ".join(INSTALL))
print("매뉴얼 목차:", " | ".join(MANUAL))
asis, tobe = 610, 58
print(f"\n[브리핑 요약] 24개 부대 결산 보고서를 {run_ms:.0f}ms에 자동 생성(수작업 가정 {asis}분 -> 검토 포함 {tobe}분, {1 - tobe / asis:.0%} 절감 가정값).")
print(f"자료 신뢰도 {rel}점, 이상 급증 후보 {len(anom)}건은 담당자 확인 대상, 보고서 24건 전부 보급관 -> 군수과장 결재 대기(자동 발간 없음).")
print(f"자체 점검 {npass}/{len(checks)} PASS. 목표(원클릭 설치·실행 완비, 매뉴얼 100%): {'달성' if npass == len(checks) else '미달'}")


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[0] 환경 점검: Python 3.11, 필수 모듈 5/5 확인 [교육용가상자료]
[1] 수집 264행(CSV+JSON 혼합), 검증 위반 0건, 이상 급증 후보 9건, 자료 신뢰도 100.0점
[2] 부대별 결산 보고서(상위 5개 부대) - 모든 보고서는 결재 대기 상태
       issue close     status
가상01대대  1147  8915  결재 대기(기안)
가상02대대   916  9139  결재 대기(기안)
가상03대대   778  7428  결재 대기(기안)
가상04대대   882  8357  결재 대기(기안)
가상05대대   920  7718  결재 대기(기안)
[3] 분류별 불출·소모 합계: {'물자불출': 8270, '수리부속': 8184, '탄약정산': 6743}
[4] 단계별 실행 로그:
   - 수집·파싱    OK    19.0ms
   - 검증       OK    8.2ms
   - 집계       OK    13.2ms
   - 이상 점검    OK    23.8ms
   - 신뢰도      OK    0.0ms
   - 보고서 생성   OK    33.2ms

[5] 자체 점검
   PASS  환경 점검 5/5
   PASS  모든 단계 OK
   PASS  24개 부대 보고서 생성
   PASS  자동 발간 0건
   PASS  원장 산식 위반 0건
   PASS  실행 시간 1분 이내
   PASS  매뉴얼 9장 작성(작성률 100%)
   PASS  설치 절차 5단계 완비
설치 스크립트 개요: 1. 가상 환경 생성 및 필수 모듈 설치(numpy, pandas, matplotlib) / 2. 환경 점검: 모듈·버전 확인 / 3. 샘플 자료로 시험 실행 / 4. 실행: run_monthly_close() / 5. 결과 확인: 보고서 24건, 결재 대기 목록
매뉴얼 목차: 1. 개요·설치 | 2. 입력 자료 양식 | 3. 실행 방법(원클릭) | 4. 검증 룰 목록 | 5. 이상 항목 처리 | 6. 결재선·권한 | 7. 오류 메시지 조치 | 8. 배치 운영 | 9. 개선 이력·문의

[브리핑 요약] 24개 부대 결산 보고서를 97ms에 자동 생성(수작업 가정 610분 -> 검토 포함 58분, 90% 절감 가정값).
자료 신뢰도 100.0점, 이상 급증 후보 9건은 담당자 확인 대상, 보고서 24건 전부 보급관 -> 군수과장 결재 대기(자동 발간 없음).
자체 점검 8/8 PASS. 목표(원클릭 설치·실행 완비, 매뉴얼 100%): 달성
```
</details>
